In [1]:
"""
Test script for LMaaS
"""
from openai import AzureOpenAI
from openai.types.chat import ChatCompletionSystemMessageParam, ChatCompletionUserMessageParam

import config
from idam_token_generator import IDAMTokenGenerator


idam = IDAMTokenGenerator(
    config.IDAM_TOKEN_ENDPOINT,
    config.IDAM_APP_CLIENT_ID,
    config.IDAM_APP_CLIENT_SECRET,
    config.IDAM_LMAAS_APP_AUDIENCE
)


llm = AzureOpenAI(
        azure_endpoint = config.OPENAI_ENDPOINT,
        azure_deployment = config.OPENAI_DEPLOYMENT_MODEL,
        api_version = config.OPENAI_AZURE_API_VERSION,
        azure_ad_token = idam.get_idam_token()
    )


Expiry_time: %s 1762224606
exp_time : %s 2025-11-04 02:50:06+00:00
current_time : %s 2025-11-04 19:51:41.969273+00:00
JWT token is NOT VALID
Generating new token.


/qumulo/shared_data/aofei_summer/miniconda3/lib/python3.13/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'idam.gehealthcloud.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


IDAM Access Token is generated


/qumulo/shared_data/aofei_summer/miniconda3/lib/python3.13/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'idam.gehealthcloud.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


IDAM Exchange Access Token is generated


In [2]:
messages = [
    ChatCompletionSystemMessageParam(role="system", content="You are a helpful assistant."),
    ChatCompletionUserMessageParam(role="user", content="What is the capital of France?"),
]

response = llm.chat.completions.create(
        model = config.OPENAI_DEPLOYMENT_MODEL,
        messages = messages,
    )

print(response.choices[0].message.content)

The capital of France is Paris.


In [4]:
# load the annotations data
import json
annotation_path01 = "/qumulo/shared_data/aofei_summer/RegTok/data/RegAlign_data_12k.json"

annotations = []
# load jsonlines
with open(annotation_path01, 'r', encoding='utf-8') as f:
    annotations = json.load(f)

In [6]:
len(annotations)

11990

In [7]:
breast_data = "/qumulo/shared_data/aofei_summer/RegTok/data/RegAlign_data_breast.json"
with open(breast_data, 'r', encoding='utf-8') as f:
    breast_annotations = json.load(f)

In [8]:
len(breast_annotations)

519

In [11]:
ids = []
for i in range(len(annotations)):
    if "BreastUS" in annotations[i]['image_file']:
        # ids.append(annotations[i]['image_file'])
        ids.append(i)
ids[0], ids[-1]

(1000, 1518)

In [12]:
annotations[ids[0]:ids[1]] = breast_annotations

In [14]:
annotations[1010]

{'image_id': '/qumulo/shared_data/aofei_summer/data/BiomedParse/BreastUS/BreastUS/train_417',
 'image_file': '/qumulo/shared_data/aofei_summer/data/BiomedParse/BreastUS/BreastUS/train/malignant (159)_ultrasound_breast.png',
 'mask_annotations': [{'mask_file': '/qumulo/shared_data/aofei_summer/data/BiomedParse/BreastUS/BreastUS/train_mask/malignant (159)_ultrasound_breast_malignant+tumor.png',
   'area': 193500,
   'iscrowd': 0,
   'image_id': '/qumulo/shared_data/aofei_summer/data/BiomedParse/BreastUS/BreastUS/train_417',
   'bbox': [379, 131, 447, 584],
   'category_id': 10,
   'id': 834,
   'slice_ratio': 1.0,
   'file_name': '/qumulo/shared_data/aofei_summer/data/BiomedParse/BreastUS/BreastUS/train/malignant (159)_ultrasound_breast.png',
   'split': 'train',
   'sentences': [{'raw': 'malignant tumor in breast ultrasound',
     'sent': 'malignant tumor in breast ultrasound',
     'sent_id': 1822}],
   'sent_ids': [1822],
   'ann_id': 834,
   'ref_id': 834,
   'modality_label': 2,
   

In [15]:
# save the new annotations data
new_annotation_path = "/qumulo/shared_data/aofei_summer/RegTok/data/RegAlign_data_12k.json"
with open(new_annotation_path, 'w', encoding='utf-8') as f:
    json.dump(annotations, f)

In [13]:
len(set([item['image_id'] for item in annotations]))

11990

In [54]:
dataset_names = set()
dataset_size = dict()
for i in annotations:
    dataset_name = i['image_id'].split("/")[7]
    if dataset_name == "MSD":
        dataset_name = i['image_id'].split("/")[8]
    dataset_names.add(dataset_name)
    dataset_size[dataset_name] = dataset_size.get(dataset_name, 0) + 1
len(dataset_names), dataset_size

(36,
 {'amos22': 3000,
  'BreastUS': 519,
  'CAMUS': 1000,
  'CDD-CESM': 1016,
  'CXR_Masks_and_Labels': 544,
  'DRIVE': 15,
  'FH-PS-AOP': 1000,
  'G1020': 816,
  'GlaS': 123,
  'ISIC': 1000,
  'kits23': 500,
  'LGG': 1014,
  'LIDC-IDRI': 1000,
  'LiverUS': 30,
  'MMs': 500,
  'Task01_BrainTumour': 1000,
  'Task02_Heart': 959,
  'Task03_Liver': 500,
  'Task04_Hippocampus': 500,
  'Task05_Prostate': 728,
  'Task06_Lung': 500,
  'Task07_Pancreas': 500,
  'Task08_HepaticVessel': 500,
  'Task09_Spleen': 500,
  'Task10_Colon': 500,
  'NeoPolyp': 800,
  'OCT-CME': 600,
  'PanNuke': 500,
  'PolypGen': 500,
  'COVID-19_CT': 100,
  'COVID-QU-Ex': 100,
  'QaTa-COV19': 100,
  'Radiography': 200,
  'REFUGE': 800,
  'siim-acr-pneumothorax': 500,
  'UWaterlooSkinCancer': 165})

In [57]:
dataset_Sample_dict = {'amos22': 1000,
 'BreastUS': 519,
 'CAMUS': 500,
 'CDD-CESM': 600,
 'CXR_Masks_and_Labels': 544,
 'DRIVE': 15,
 'FH-PS-AOP': 500,
 'G1020': 816,
 'GlaS': 123,
 'ISIC': 500,
 'kits23': 100,
 'LGG': 500,
 'LIDC-IDRI': 500,
 'LiverUS': 30,
 'MMs': 500,
 'Task01_BrainTumour': 400,
 'Task02_Heart': 500,
 'Task03_Liver': 200,
 'Task04_Hippocampus': 200,
 'Task05_Prostate': 228,
 'Task06_Lung': 300,
 'Task07_Pancreas': 200,
 'Task08_HepaticVessel': 200,
 'Task09_Spleen': 200,
 'Task10_Colon': 200,
 'NeoPolyp': 400,
 'OCT-CME': 600,
 'PanNuke': 300,
 'PolypGen': 300,
 'COVID-19_CT': 50,
 'COVID-QU-Ex': 50,
 'QaTa-COV19': 50,
 'Radiography': 200,
 'REFUGE': 400,
 'siim-acr-pneumothorax': 100,
 'UWaterlooSkinCancer': 165}

# Sample annotations per dataset based on dataset_Sample_dict
import random
random.seed(42)

def _get_dataset_name(img_path: str):
    parts = img_path.split('/')
    ds = parts[7] if len(parts) > 7 else 'UNKNOWN'
    if ds == 'MSD' and len(parts) > 8:
        ds = parts[8]
    return ds

# Build mapping: dataset -> indices
dataset_to_indices = {}
for idx, ann in enumerate(annotations):
    img_id = ann.get('image_id', '')
    ds = _get_dataset_name(img_id) if isinstance(img_id, str) else 'UNKNOWN'
    dataset_to_indices.setdefault(ds, []).append(idx)

# Perform sampling
sampled_indices = []
sampled_counts = {}
for ds, idxs in dataset_to_indices.items():
    k_req = dataset_Sample_dict.get(ds, len(idxs))
    k = min(k_req, len(idxs))
    chosen = idxs if k == len(idxs) else random.sample(idxs, k)
    sampled_indices.extend(chosen)
    sampled_counts[ds] = len(chosen)

# Keep stable order (optional)
sampled_indices.sort()

# Overwrite annotations with the sampled subset
annotations_sampled = [annotations[i] for i in sampled_indices]
# annotations = annotations_sampled

# Quick sanity check
print('Total sampled:', len(annotations_sampled))
print('Per-dataset sampled counts:', {k: sampled_counts[k] for k in sorted(sampled_counts)})

Total sampled: 11990
Per-dataset sampled counts: {'BreastUS': 519, 'CAMUS': 500, 'CDD-CESM': 600, 'COVID-19_CT': 50, 'COVID-QU-Ex': 50, 'CXR_Masks_and_Labels': 544, 'DRIVE': 15, 'FH-PS-AOP': 500, 'G1020': 816, 'GlaS': 123, 'ISIC': 500, 'LGG': 500, 'LIDC-IDRI': 500, 'LiverUS': 30, 'MMs': 500, 'NeoPolyp': 400, 'OCT-CME': 600, 'PanNuke': 300, 'PolypGen': 300, 'QaTa-COV19': 50, 'REFUGE': 400, 'Radiography': 200, 'Task01_BrainTumour': 400, 'Task02_Heart': 500, 'Task03_Liver': 200, 'Task04_Hippocampus': 200, 'Task05_Prostate': 228, 'Task06_Lung': 300, 'Task07_Pancreas': 200, 'Task08_HepaticVessel': 200, 'Task09_Spleen': 200, 'Task10_Colon': 200, 'UWaterlooSkinCancer': 165, 'amos22': 1000, 'kits23': 100, 'siim-acr-pneumothorax': 100}


In [67]:
# len(dataset_Sample_dict), len(annotations_sampled), annotations_sampled[1]

In [59]:
annotations = annotations_sampled

In [60]:
len(annotations)

11990

In [61]:
# save annotations
annotation_path_save = "/qumulo/shared_data/aofei_summer/RegTok/data/RegAlign_data_12k.json"
with open(annotation_path_save, "w") as f:
    json.dump(annotations, f)

In [68]:
breast_annotations = []
for i in annotations[1000:1519]:
    i_copy = i.copy()
    i_copy['mask_annotations'] = i_copy['mask_annotations'][:1]
    breast_annotations.append(i_copy)

In [69]:
breast_annotations[0]

{'image_id': '/qumulo/shared_data/aofei_summer/data/BiomedParse/BreastUS/BreastUS/train_393',
 'image_file': '/qumulo/shared_data/aofei_summer/data/BiomedParse/BreastUS/BreastUS/train/malignant (137)_ultrasound_breast.png',
 'mask_annotations': [{'mask_file': '/qumulo/shared_data/aofei_summer/data/BiomedParse/BreastUS/BreastUS/train_mask/malignant (137)_ultrasound_breast_malignant+tumor.png',
   'area': 314917,
   'iscrowd': 0,
   'image_id': '/qumulo/shared_data/aofei_summer/data/BiomedParse/BreastUS/BreastUS/train_393',
   'bbox': [39, 187, 712, 643],
   'category_id': 10,
   'id': 786,
   'slice_ratio': 1.0,
   'file_name': '/qumulo/shared_data/aofei_summer/data/BiomedParse/BreastUS/BreastUS/train/malignant (137)_ultrasound_breast.png',
   'split': 'train',
   'sentences': [{'raw': 'malignant tumor in breast ultrasonogram',
     'sent': 'malignant tumor in breast ultrasonogram',
     'sent_id': 1716},
    {'raw': 'malignant tumor in breast ultrasound',
     'sent': 'malignant tumor 

In [70]:
annotation_path_save = "/qumulo/shared_data/aofei_summer/RegTok/data/RegAlign_data_breast.json"
with open(annotation_path_save, "w") as f:
    json.dump(breast_annotations, f)

{'image_id': '/qumulo/shared_data/aofei_summer/data/BiomedParse/BreastUS/BreastUS/train_180',
 'image_file': '/qumulo/shared_data/aofei_summer/data/BiomedParse/BreastUS/BreastUS/train/benign (261)_ultrasound_breast.png',
 'mask_annotations': [{'mask_file': '/qumulo/shared_data/aofei_summer/data/BiomedParse/BreastUS/BreastUS/train_mask/benign (261)_ultrasound_breast_benign+tumor.png',
   'area': 105735,
   'iscrowd': 0,
   'image_id': '/qumulo/shared_data/aofei_summer/data/BiomedParse/BreastUS/BreastUS/train_180',
   'bbox': [140, 52, 281, 493],
   'category_id': 10,
   'id': 360,
   'slice_ratio': 1.0,
   'file_name': '/qumulo/shared_data/aofei_summer/data/BiomedParse/BreastUS/BreastUS/train/benign (261)_ultrasound_breast.png',
   'split': 'train',
   'sentences': [{'raw': 'benign tumor',
     'sent': 'benign tumor',
     'sent_id': 786}],
   'sent_ids': [786],
   'ann_id': 360,
   'ref_id': 360,
   'modality_label': 2,
   'quantizer_code': 'M2_22'},
  {'mask_file': '/qumulo/shared_dat

In [63]:
annotations[0]

{'image_id': '/qumulo/shared_data/aofei_summer/data/BiomedParse/amos22/amos22/MRI/train_877',
 'image_file': '/qumulo/shared_data/aofei_summer/data/BiomedParse/amos22/amos22/MRI/train/amos_0532_6_MRI_abdomen.png',
 'mask_annotations': [{'mask_file': '/qumulo/shared_data/aofei_summer/data/BiomedParse/amos22/amos22/MRI/train_mask/amos_0532_6_MRI_abdomen_right+kidney.png',
   'area': 3711,
   'iscrowd': 0,
   'image_id': '/qumulo/shared_data/aofei_summer/data/BiomedParse/amos22/amos22/MRI/train_877',
   'bbox': [621, 368, 76, 85],
   'category_id': 3,
   'id': 4701,
   'slice_ratio': 0.3,
   'file_name': '/qumulo/shared_data/aofei_summer/data/BiomedParse/amos22/amos22/MRI/train/amos_0532_6_MRI_abdomen.png',
   'split': 'train',
   'sentences': [{'raw': 'right kidney in abdominal magnetic resonance imaging',
     'sent': 'right kidney in abdominal magnetic resonance imaging',
     'sent_id': 9793},
    {'raw': 'right kidney in abdominal MRI',
     'sent': 'right kidney in abdominal MRI',
 

In [46]:
# annotations['/qumulo/shared_data/aofei_summer/data/BiomedParse/MSD/MSD/Task03_Liver/train_48']

In [30]:
image_H, image_W = 1024, 1024
def preprocess_annotations(annotation):
    processed = []
    processed_with_info = []
    for region in annotation['mask_annotations']:
        item = {
            "id": region['id'],
            "quantizer_code": region['quantizer_code']
        }
        item_with_info = {
            "id": region['id'],
            "quantizer_code": region['quantizer_code'],
            "mask_file": region.get("mask_file", ""),
            "image_id": region.get("image_id", "")
        }

        processed_bbox = [
            round(region['bbox'][1] / image_W, 3),
            round(region['bbox'][0] / image_H, 3),
            round(region['bbox'][3] / image_W, 3),
            round(region['bbox'][2] / image_H, 3)
        ]
        item['bbox'] = processed_bbox
        processed_sentences = []
        for sentence in region['sentences']:
            processed_sentences.append(sentence['raw'])
        item['sentences'] = processed_sentences
        processed.append(item)
        processed_with_info.append(item_with_info)
    return processed, processed_with_info

In [31]:
# preprocess_annotations(annotation=annotations['/qumulo/shared_data/aofei_summer/data/BiomedParse/MSD/MSD/Task03_Liver/train_48'])


In [32]:
sampled_image_id = 0
processed_items = []
processed_items_with_info = []
for k in annotations:
    annotation = k
    processed, processed_with_info = preprocess_annotations(annotation=annotation)
    image_masks = {
        "image_id": sampled_image_id,
        "masks": processed
    }
    processed_items.append(image_masks) 
    image_masks_info = {
        "image_id": sampled_image_id,
        "image_file": annotation.get("image_file", ""),
        "num_masks": len(processed),
        "mask_code": [(item['id'], item['quantizer_code'], item['mask_file']) for item in processed_with_info]
    }

    processed_items_with_info.append(image_masks_info)
    sampled_image_id += 1

In [33]:
processed_items[1], processed_items_with_info[1]

({'image_id': 1,
  'masks': [{'id': 44558,
    'quantizer_code': 'M0_1',
    'bbox': [0.342, 0.512, 0.073, 0.075],
    'sentences': ['aorta in abdominal computed tomography',
     'aorta in abdominal CT']},
   {'id': 44560,
    'quantizer_code': 'M0_7',
    'bbox': [0.244, 0.611, 0.11, 0.192],
    'sentences': ['spleen in abdominal CT']},
   {'id': 44561,
    'quantizer_code': 'M0_8',
    'bbox': [0.418, 0.512, 0.048, 0.069],
    'sentences': ['esophagus']},
   {'id': 44562,
    'quantizer_code': 'M0_16',
    'bbox': [0.281, 0.098, 0.444, 0.604],
    'sentences': ['liver in abdominal CT']},
   {'id': 44564,
    'quantizer_code': 'M0_8',
    'bbox': [0.34, 0.615, 0.116, 0.118],
    'sentences': ['gastric organ in abdominal CT',
     'stomach in abdominal CT',
     'stomach in abdominal computed tomography']},
   {'id': 44566,
    'quantizer_code': 'M0_28',
    'bbox': [0.398, 0.367, 0.083, 0.087],
    'sentences': ['postcava in abdominal computed tomography',
     'postcava in abdominal

In [26]:
Alignment_prompt = """
You are given region-level annotations for one or more medical images. Each image item includes:
- "image_id": an integer id,
- "masks": a list of region objects, each with:
    - "quantizer_code": a code as an identifier in LLM (e.g., M0_25),
    - "bbox": normalized bounding box [x, y, w, h] (values in [0,1]),
    - "sentences": free-text descriptions (list of strings).

Assume you can see the image implicitly and must use only the provided annotation information (quantizer codes, bboxes, and sentences) to perform the tasks below.

Task (produce a single JSON output per image):
- Generate user-assistant dialogues about the image based solely on the provided regions.

Output rules and format (strict):
- Always return a JSON list containing one object per input image: [{...}, {...}, ...].
- Each image object must contain:
  {
    "image_id": <the input image_id>,
    "dialogues": [ <list of dialogue turn objects> ]
  }
- Each dialogue turn is a JSON object:
  {
    "User": "<user utterance>",
    "Assistant": "<assistant reply>"
  }
  Optionally include "mask_ids_order": [i, j, ...] when the assistant refers to specific region masks for detection/segmentation/localization. mask_ids_order must be a list of integer indices into the input image's "masks" list (0-based) in the order the assistant mentions them.

Segmentation / Detection / Localization specific rules:
- When the user's question requests detection, segmentation, or asks to mark/segment a structure explicitly, the Assistant MUST:
  1) mention the corresponding quantizer code(s) inline in square brackets, e.g., "tumor [M0_25]".
  2) include "mask_ids_order": the list of indices (0-based) pointing to the masks in the input in the same reply object.
- If the Assistant references spatial relationships (e.g., "inside the liver"), include the region code(s).
- If the user's question does NOT require detection/segmentation/localization (e.g., modality, general findings, diagnosis, differential, measurement interpretation), **DO NOT** output quantizer codes, bboxes or mask_ids_order. Provide only clinical text.
- mask_ids_order indices correspond to the order in the provided "masks" list for that image.

Dialogue diversity and content templates for questions and answers:
- Include a mix of these user question types across dialogues:
  - Modality / Acquisition: "What modality is this?"
  - Structure identification: "Locate the liver/kidney/lung nodule.".
  - Detection / Segmentation request: "Please segment the tumor.", "Mark the lesion for segmentation."
  - Diagnosis / Impression: "What diseases are included in the picture?", "Is the lung healthy?", "What abnormality is identified? Provide the diagnosis and segment the affected area."
- When possible, produce at least one dialogue that asks for detection/segmentation and other dialogues that ask higher-level clinical interpretation.
- If there are many masks in the image, prioritize mentioning clinically significant structures (e.g., main organs, tumors, lesions) over other normal anatomy.
    -  If there are too many masks, you can mention more than 1 masks in one query.

Behavioral constraints:
- Do not hallucinate findings not supported by the provided sentences or masks. Use only provided mask descriptions and bboxes to assert presence/location of structures.
- Keep assistant replies clinically appropriate.
- When returning multiple images, the returned JSON must be a list with one object per input image, preserving input image_id values.
- ** Very Important!!!!
  - ** Do not use the words "annotated", "annotation" or "labels" to describe findings (for example, avoid phrases like "according to the annotations" or "the labels indicate"). **Assume you are viewing the image itself** and report findings directly; do not write "The label indicates" or "based on the annotation".
  - You must include negative examples, for example, you can ask a organ or disease that are not present in the given information and then response like "There is no xxx in the image, I can not segment it".

Example input of one image (for reference only):
{
  "image_id": 0,
  "masks": [
    {"quantizer_code": "M0_16", "bbox": [0.412, 0.496, 0.389, 0.225], "sentences": ["liver"]},
    {"quantizer_code": "M0_25", "bbox": [0.555, 0.658, 0.046, 0.046], "sentences": ["tumor"]}
  ]
}

Example output (for one image):
[
  {
    "image_id": 0,
    "dialogues": [
      {
        "User": "Can you segment the liver in this CT scan?",
        "Assistant": "Yes. The liver is segmented as [M0_16].",
        "mask_ids_order": [0]
      },
      {
        "User": "Can you also segment the spleen in this image?",
        "Assistant": "No, there is no spleen in the image."
      },
      {
        "User": "Is the liver healthy?",
        "Assistant": "No, the findings suggest a focal hepatic lesion suspicious for a primary tumor."
      },
      {
        "User": "What abnormality is identified on the liver? Provide the diagnosis and segment the affected area.",
        "Assistant": "Findings consistent with liver tumor (cancer) are present, segmented as [M0_25].",
        "mask_ids_order": [1]
      },
      
    ]
  }
]
## **Do not use the words** "annotated", "annotation" or "labels" to describe findings (for example, avoid phrases like "according to the annotations" or "the labels indicate"). **Assume you are viewing the image itself** and report findings directly; do not write "The label indicates" or "based on the annotation".
## Do not use \u2011 to represent -, use - directly. For example, when asked the modality of X-ray, you can say X-ray or Xray but no \u2011.
## you must record then mask_ids_order correctly so that we can match the codes with mask labels.
## Also do not use words like "in the descriptions", "as described", "report", just image you are seeing the image. If you are unsure or feel there are conflicts in the annotations, simply generate segmentation prompts and avoid generating disgnosis or other type of questions.

Finally: The API will provide the "masks" list as input. Produce the JSON outputs (one object per image) strictly following the rules above. Do not include any extra text outside the JSON list in the model's final reply. """


In [52]:
list_outputs = []
# list_outputs = list_outputs[:100]

In [27]:
output_json_file = "RegAlign_GPT5_mini_v9.0.jsonl"
ans_file = open(output_json_file, "a")

In [34]:
len(processed_items)

21429

In [35]:
batch_size = 10
max_retry = 3
from tqdm import tqdm
# for i in tqdm(range(10, len(processed_items), batch_size)):
for i in tqdm(range(0, 10, batch_size)):
    num_try = 1
    items = processed_items[i:i + batch_size]
    before_process_items = processed_items_with_info[i:i + batch_size]
    messages = [
        ChatCompletionSystemMessageParam(role="system", content="You are a helpful assistant." + Alignment_prompt + "\n\n"),
        ChatCompletionUserMessageParam(role="user", content="The input with multiple images:" + str(items)),
    ]
    try:
        response = llm.chat.completions.create(
                model = config.OPENAI_DEPLOYMENT_MODEL,
                messages = messages,
            )
    except:
        idam = IDAMTokenGenerator(
            config.IDAM_TOKEN_ENDPOINT,
            config.IDAM_APP_CLIENT_ID,
            config.IDAM_APP_CLIENT_SECRET,
            config.IDAM_LMAAS_APP_AUDIENCE
        )


        llm = AzureOpenAI(
                azure_endpoint = config.OPENAI_ENDPOINT,
                azure_deployment = config.OPENAI_DEPLOYMENT_MODEL,
                api_version = config.OPENAI_AZURE_API_VERSION,
                azure_ad_token = idam.get_idam_token()
            )
        response = llm.chat.completions.create(
                model = config.OPENAI_DEPLOYMENT_MODEL,
                messages = messages,
            )
        num_try += 1
        if num_try > max_retry:
            print(f"Max retries exceeded for batch starting at index {i}")
            continue

    # print(response.choices[0].message.content)
    llm_out = response.choices[0].message.content
    llm_out_json = json.loads(llm_out)
    items = processed_items[i:i + batch_size]
    original_items = processed_items_with_info[i:i + batch_size]
    for j in range(batch_size):
        original_item = original_items[j]
        llm_out_json[j]['image_file'] = original_item['image_file']
        llm_out_json[j]['mask_code'] = original_item['mask_code']
    list_outputs.extend(llm_out_json)
    ans_file.write("\n".join([json.dumps(x) for x in llm_out_json]) + "\n")
    ans_file.flush()
# ans_file.close()


  0%|          | 0/1 [00:00<?, ?it/s]

Expiry_time: %s 1762218694
exp_time : %s 2025-11-04 01:11:34+00:00
current_time : %s 2025-11-04 01:38:54.828039+00:00
JWT token is NOT VALID
Generating new token.


/qumulo/shared_data/aofei_summer/miniconda3/lib/python3.13/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'idam.gehealthcloud.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


IDAM Access Token is generated


/qumulo/shared_data/aofei_summer/miniconda3/lib/python3.13/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'idam.gehealthcloud.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


IDAM Exchange Access Token is generated


  0%|          | 0/1 [00:55<?, ?it/s]


NameError: name 'list_outputs' is not defined

In [56]:
# llm_out = response.choices[0].message.content
# llm_out_json = json.loads(llm_out)
# # post-processing the llm outputs
# items = processed_items[i:i + batch_size]
# original_items = processed_items_with_info[i:i + batch_size]
# for j in range(batch_size):
#     original_item = original_items[j]
#     llm_out_json[j]['image_file'] = original_item['image_file']
#     llm_out_json[j]['mask_code'] = original_item['mask_code']

In [36]:
llm_out_json
##
## do not mention words "annotation" or "labels" like this: according to the annotations (the labels), you should assume you are seeing the image itself and do not provide information like "The label indicates", "based on the annotation".
# 


[{'image_id': 0,
  'dialogues': [{'User': 'What modality is this image?',
    'Assistant': 'This is an abdominal MRI.'},
   {'User': 'Please segment the right kidney.',
    'Assistant': 'Right kidney segmented as [M1_3].',
    'mask_ids_order': [0]},
   {'User': 'Can you segment the spleen here?',
    'Assistant': 'There is no spleen in the image, I cannot segment it.'},
   {'User': 'What structures are visible and is there any obvious abnormality?',
    'Assistant': 'Visible structures include both kidneys, the aorta, and the postcava. No focal abnormality is evident from the provided view.'}],
  'image_file': '/qumulo/shared_data/aofei_summer/data/BiomedParse/amos22/amos22/MRI/train/amos_0532_6_MRI_abdomen.png',
  'mask_code': [(4701,
    'M1_3',
    '/qumulo/shared_data/aofei_summer/data/BiomedParse/amos22/amos22/MRI/train_mask/amos_0532_6_MRI_abdomen_right+kidney.png'),
   (4702,
    'M1_11',
    '/qumulo/shared_data/aofei_summer/data/BiomedParse/amos22/amos22/MRI/train_mask/amos_0